# 09 Generate GOBM Stratified-Domain ID, Weights, and DII Intermediates

Purpose: generate the North/South stratified GOBM inputs needed by `SamplingDomains.png`.

Inputs:
- GOBM `.pkl` files under `gs://leap-persistent/vacquaviva/GOBMs/` or an equivalent local directory

Outputs used directly by `scripts/plot_stratified_sampling_domains.py`:
- `south_dim_all.csv`
- `south_scales_all.csv`
- `north_dim_all.csv`
- `north_scales_all.csv`
- `Finalimbs_sampled16k_South_<model>.txt`
- `Finalimbs_sampled16k_North_<model>.txt`

Additional provenance outputs:
- `south_dim_input.csv`
- `south_scales_input.csv`
- `north_dim_input.csv`
- `north_scales_input.csv`
- `Finalweights_sampled16k_South_<model>.txt`
- `Finalweights_sampled16k_North_<model>.txt`

where `<model>` is `ETHZ`, `FESOM`, `IPSL`, `MRI`, and `NorESM`.

Estimated runtime: very slow. Each model has two 16k stratified domains; each domain computes intrinsic dimension and runs DADAPy backward greedy DII elimination.

Notes:
- This notebook combines the stratified intrinsic-dimension and DII/weight calculations in one workflow.
- This optional regeneration step requires large upstream GOBM `.pkl` files and a compatible Python environment.
- The South domain is latitude -60 to -30. The North domain is latitude 30 to 60.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from dadapy import Data
from dadapy.feature_weighting import FeatureWeighting
from sklearn.preprocessing import StandardScaler

# Use the GCS root by default. Set this to a local folder if the pkl files have been downloaded.
GOBM_ROOT = "gs://leap-persistent/vacquaviva/GOBMs"
OUTPUT_DIR = Path("../intermediates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLE = 16000
RANDOM_STATE = 13
N_EPOCHS = 80
TIME_MIN = "2020-01-01"
MAXK_CAP = 10000
RANGE_MAX = 1024


## Model Files and Feature Definitions


In [ ]:
MODEL_FILES = {
    "ETHZ": "CESM_ETHZ/MLinput_CESM_ETHZ_mon_1x1_197001_202212.pkl",
    "FESOM": "FESOM2_REcoM/MLinput_FESOM2_REcoM_mon_1x1_197001_202212.pkl",
    "IPSL": "IPSL/MLinput_IPSL_mon_1x1_197001_202212.pkl",
    "MRI": "MRI_ESM2_2/MLinput_MRI_ESM2_2_mon_1x1_197001_202212.pkl",
    "NorESM": "NorESM/MLinput_NorESM_mon_1x1_197001_202212.pkl",
}

FEATURES = [
    "sst",
    "sst_anom",
    "sss",
    "sss_anom",
    "chl_log",
    "chl_log_anom",
    "mld_log",
    "xco2",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"
SOCAT_MASK = "socat_mask"

DOMAINS = {
    "South": slice(-60, -30),
    "North": slice(30, 60),
}


## Helper Functions


In [ ]:
def model_path(model):
    root = str(GOBM_ROOT).rstrip("/")
    return f"{root}/{MODEL_FILES[model]}"

def load_model_frame(model):
    print(f"Loading {model}")
    frame = pd.read_pickle(model_path(model))
    frame[TARGET] = frame["sfco2"] - frame["xco2"]
    frame = frame[FEATURES + [SOCAT_MASK, TARGET]]
    frame = frame[frame.index.get_level_values("time") >= TIME_MIN]
    return frame

def select_domain(frame, domain):
    latitude_slice = DOMAINS[domain]
    domain_frame = frame.loc[pd.IndexSlice[:, :, latitude_slice], :]
    domain_frame = domain_frame.drop_duplicates(subset=[TARGET]).dropna()
    return domain_frame

def sample_and_scale(domain_frame):
    sample = domain_frame.sample(n=N_SAMPLE, random_state=RANDOM_STATE)
    scaled = StandardScaler().fit_transform(sample.loc[:, FEATURES + [TARGET]])
    return pd.DataFrame(scaled, columns=FEATURES + [TARGET])

def compute_id_tables(scaled):
    data_all = Data(scaled.to_numpy(), verbose=False)
    data_all.compute_distances(maxk=min(scaled.shape[0] - 1, MAXK_CAP))
    ids_all, _, scales_all = data_all.return_id_scaling_gride(range_max=RANGE_MAX)

    input_scaled = scaled.drop([TARGET], axis=1)
    data_input = Data(input_scaled.to_numpy(), verbose=False)
    data_input.compute_distances(maxk=min(input_scaled.shape[0] - 1, MAXK_CAP))
    ids_input, _, scales_input = data_input.return_id_scaling_gride(range_max=RANGE_MAX)

    return ids_all, ids_input, scales_all, scales_input

def run_weight_dii(model, domain, scaled):
    print(f"Generating DII/weights for {model} {domain}")
    target = FeatureWeighting(scaled[[TARGET]].to_numpy(), verbose=True)
    inputs = FeatureWeighting(scaled[FEATURES].to_numpy(), verbose=True)

    final_imbs, final_weights = inputs.return_backward_greedy_dii_elimination(
        target_data=target,
        initial_weights=None,
        n_epochs=N_EPOCHS,
        learning_rate=None,
        decaying_lr="cos",
    )

    np.savetxt(OUTPUT_DIR / f"Finalweights_sampled16k_{domain}_{model}.txt", final_weights)
    np.savetxt(OUTPUT_DIR / f"Finalimbs_sampled16k_{domain}_{model}.txt", final_imbs)


## Generate Stratified ID and DII/Weight Intermediates

Slow cell.


In [ ]:
south_dim_all = pd.DataFrame()
south_dim_input = pd.DataFrame()
south_scales_all = pd.DataFrame()
south_scales_input = pd.DataFrame()

north_dim_all = pd.DataFrame()
north_dim_input = pd.DataFrame()
north_scales_all = pd.DataFrame()
north_scales_input = pd.DataFrame()

for model in MODEL_FILES:
    frame = load_model_frame(model)

    for domain in ["South", "North"]:
        domain_frame = select_domain(frame, domain)
        print(model, domain, "rows:", domain_frame.shape[0])
        scaled = sample_and_scale(domain_frame)

        ids_all, ids_input, scales_all, scales_input = compute_id_tables(scaled)
        run_weight_dii(model, domain, scaled)

        if domain == "South":
            south_dim_all[model] = ids_all
            south_dim_input[model] = ids_input
            south_scales_all[model] = scales_all
            south_scales_input[model] = scales_input
        else:
            north_dim_all[model] = ids_all
            north_dim_input[model] = ids_input
            north_scales_all[model] = scales_all
            north_scales_input[model] = scales_input

south_dim_all.to_csv(OUTPUT_DIR / "south_dim_all.csv", index=False)
south_dim_input.to_csv(OUTPUT_DIR / "south_dim_input.csv", index=False)
south_scales_all.to_csv(OUTPUT_DIR / "south_scales_all.csv", index=False)
south_scales_input.to_csv(OUTPUT_DIR / "south_scales_input.csv", index=False)

north_dim_all.to_csv(OUTPUT_DIR / "north_dim_all.csv", index=False)
north_dim_input.to_csv(OUTPUT_DIR / "north_dim_input.csv", index=False)
north_scales_all.to_csv(OUTPUT_DIR / "north_scales_all.csv", index=False)
north_scales_input.to_csv(OUTPUT_DIR / "north_scales_input.csv", index=False)


## Outputs Created

Quick check:


In [ ]:
expected_outputs = [
    "south_dim_all.csv",
    "south_dim_input.csv",
    "south_scales_all.csv",
    "south_scales_input.csv",
    "north_dim_all.csv",
    "north_dim_input.csv",
    "north_scales_all.csv",
    "north_scales_input.csv",
]

for model in MODEL_FILES:
    for domain in ["South", "North"]:
        expected_outputs.extend([
            f"Finalweights_sampled16k_{domain}_{model}.txt",
            f"Finalimbs_sampled16k_{domain}_{model}.txt",
        ])

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
